In [ ]:
#@title Cell 01.1 - Notebook overview
# This cell fixes the purpose, inputs, analysis question and 9-cell structure of Notebook 01.

from IPython.display import display, Markdown

display(Markdown(r"""
# Notebook 01: blaTEM-1 high-MIC chromosomal clustering

## Purpose
Determine whether the 16 upper-MIC pathogens within the 176 blaTEM-1-only *E. coli* pathogens are concentrated in a closely related chromosomal background or distributed across multiple backgrounds.

## Fixed inputs
Notebook 01 reuses the previous `Genome_MIC_AMR_Emergence` outputs:

- ceftazidime pathogen index and observed log2(MIC);
- candidate acquired beta-lactamase presence/absence;
- Notebook 04 genome-wide chromosomal relatedness matrix K.

The previous project is read only. All new outputs are written to `Ceftazidime_Chromosomal_Evolution`.

## Fixed cohort rule
`blaTEM-1 only` means blaTEM-1 is present and no other candidate acquired beta-lactamase in the Notebook 05 panel is present.

Upper-MIC outliers are reconstructed on the log2(MIC) scale using the previous rule: values above Q3 + 1.5 x IQR.

Expected counts are 176 blaTEM-1-only pathogens, including 16 upper-MIC outliers and 160 remaining pathogens.

## Main test
The mean pairwise K relatedness among the 16 upper-MIC pathogens is compared with random groups of 16 drawn from the same 176-pathogen cohort.

Notebook 01 contains 9 code cells.
"""))

print(
    'Notebook 01 overview complete.\n'
    'Transition: Cell 01.2 will locate the public repository, locate both project folders and validate the fixed input paths.'
)


In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 01.2 - Mount Drive and locate fixed inputs
# This cell defines the previous-project read-only inputs and the new-project output directories.

from pathlib import Path
import hashlib
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform
from IPython.display import display
PREVIOUS_PROJECT_ROOT = _previous_project_root(PROJECT_ROOT)
PROJECT_ROOT = _repo_root()

# Previous-project source directories.
NOTEBOOK02_DIRECTORY = PREVIOUS_PROJECT_ROOT / '02_Data_Preparation' / 'Notebook02'
NOTEBOOK03_DIRECTORY = PREVIOUS_PROJECT_ROOT / '02_Data_Preparation' / 'Notebook03'
NOTEBOOK04_DIRECTORY = PREVIOUS_PROJECT_ROOT / '04_Population_Structure' / 'Notebook04'

# Fixed inputs from the previous project.
ACQUIRED_PRESENCE_PATH = (
    NOTEBOOK02_DIRECTORY
    / '02_ceftazidime_candidate_acquired_beta_lactamase_presence.csv.gz'
)
PATHOGEN_INDEX_PATH = (
    NOTEBOOK03_DIRECTORY
    / '03_ceftazidime_pathogen_index.csv'
)
RELATEDNESS_MATRIX_PATH = (
    NOTEBOOK04_DIRECTORY
    / '04_genome_wide_relatedness_matrix.npz'
)
NOTEBOOK04_QC_PATH = (
    NOTEBOOK04_DIRECTORY
    / '04_population_structure_qc_summary.csv'
)

# New-project output directories.
HIGH_MIC_DIRECTORY = PROJECT_ROOT / '02_Data_New' / 'High_MIC_16'
COMPARISON_DIRECTORY = PROJECT_ROOT / '02_Data_New' / 'Comparison_160'
RELATEDNESS_DIRECTORY = PROJECT_ROOT / '04_Intermediate' / 'Relatedness'
TABLE_DIRECTORY = PROJECT_ROOT / '05_Results' / 'Tables'
FIGURE_DIRECTORY = PROJECT_ROOT / '05_Results' / 'Figures'
STATISTICAL_OUTPUT_DIRECTORY = PROJECT_ROOT / '05_Results' / 'Statistical_Outputs'

for directory in [
    HIGH_MIC_DIRECTORY,
    COMPARISON_DIRECTORY,
    RELATEDNESS_DIRECTORY,
    TABLE_DIRECTORY,
    FIGURE_DIRECTORY,
    STATISTICAL_OUTPUT_DIRECTORY,
]:
    directory.mkdir(parents=True, exist_ok=True)

required_inputs = [
    ACQUIRED_PRESENCE_PATH,
    PATHOGEN_INDEX_PATH,
    RELATEDNESS_MATRIX_PATH,
    NOTEBOOK04_QC_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        'Required previous-project input(s) were not found:\n'
        + '\n'.join(missing_inputs)
    )


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


print(f'Previous project: {PREVIOUS_PROJECT_ROOT}')
print(f'New project:      {PROJECT_ROOT}')
print('All fixed input paths were found.')
print('Previous-project files will be treated as read only.')
print(
    'Transition: Cell 01.3 will load, align and validate the phenotype, '
    'acquired-gene and relatedness inputs.'
)


In [ ]:
#@title Cell 01.3 - Load, align and validate fixed inputs
# This cell aligns the previous-project inputs by Notebook 03 row index and BioSample.

EXPECTED_PATHOGENS = 1672
EXPECTED_RELATEDNESS_SNPS = 27988

pathogen_index = pd.read_csv(
    PATHOGEN_INDEX_PATH,
    dtype={
        'biosample': str,
        'assembly_accession': str,
    },
)

acquired_presence_raw = pd.read_csv(
    ACQUIRED_PRESENCE_PATH,
    dtype={'biosample': str},
)

relatedness_archive = np.load(RELATEDNESS_MATRIX_PATH)
K = relatedness_archive['relatedness_matrix'].astype(np.float64)
relatedness_rows = relatedness_archive['notebook03_row_index'].astype(np.int64)

notebook04_qc = pd.read_csv(NOTEBOOK04_QC_PATH)

# ------------------------------------------------------------
# Validate pathogen index and phenotype.
# ------------------------------------------------------------

if len(pathogen_index) != EXPECTED_PATHOGENS:
    raise ValueError(
        f'Expected {EXPECTED_PATHOGENS} pathogen rows; found {len(pathogen_index)}.'
    )

required_pathogen_columns = {
    'notebook03_row_index',
    'biosample',
    'assembly_accession',
    'log2_mic',
}

missing_columns = required_pathogen_columns - set(pathogen_index.columns)
if missing_columns:
    raise ValueError(
        f'Pathogen index is missing required column(s): {sorted(missing_columns)}'
    )

pathogen_index = (
    pathogen_index
    .sort_values('notebook03_row_index')
    .reset_index(drop=True)
)

if not np.array_equal(
    pathogen_index['notebook03_row_index'].to_numpy(dtype=np.int64),
    np.arange(EXPECTED_PATHOGENS, dtype=np.int64),
):
    raise ValueError('Notebook 03 row indices are not the expected 0..1671 order.')

if pathogen_index['biosample'].duplicated().any():
    raise ValueError('Pathogen index contains duplicate BioSamples.')

pathogen_index['log2_mic'] = pd.to_numeric(
    pathogen_index['log2_mic'],
    errors='raise',
)

if not np.isfinite(pathogen_index['log2_mic'].to_numpy(dtype=float)).all():
    raise ValueError('log2(MIC) contains a missing or non-finite value.')

# ------------------------------------------------------------
# Align acquired beta-lactamase presence by BioSample.
# ------------------------------------------------------------

if 'biosample' not in acquired_presence_raw.columns:
    raise ValueError('Acquired-gene presence file does not contain biosample.')

if acquired_presence_raw['biosample'].duplicated().any():
    raise ValueError('Acquired-gene presence file contains duplicate BioSamples.')

acquired_presence = pathogen_index[['biosample']].merge(
    acquired_presence_raw,
    on='biosample',
    how='left',
    validate='one_to_one',
)

if acquired_presence.drop(columns=['biosample']).isna().any().any():
    raise ValueError(
        'At least one pathogen is missing from the acquired-gene presence matrix.'
    )

acquired_gene_names = [
    column
    for column in acquired_presence.columns
    if column != 'biosample'
]

for gene in acquired_gene_names:
    acquired_presence[gene] = pd.to_numeric(
        acquired_presence[gene],
        errors='raise',
    ).astype(np.uint8)

    observed = set(acquired_presence[gene].unique().tolist())
    if not observed.issubset({0, 1}):
        raise ValueError(
            f'Acquired gene {gene} is not binary presence/absence: {sorted(observed)}'
        )

# ------------------------------------------------------------
# Validate K and its provenance.
# ------------------------------------------------------------

if K.shape != (EXPECTED_PATHOGENS, EXPECTED_PATHOGENS):
    raise ValueError(
        f'Expected K shape {(EXPECTED_PATHOGENS, EXPECTED_PATHOGENS)}; found {K.shape}.'
    )

if not np.array_equal(
    relatedness_rows,
    np.arange(EXPECTED_PATHOGENS, dtype=np.int64),
):
    raise ValueError('K row order does not match Notebook 03 row identifiers.')

if not np.isfinite(K).all():
    raise ValueError('K contains a missing or non-finite value.')

if not np.allclose(K, K.T, rtol=1e-10, atol=1e-10):
    raise ValueError('K is not symmetric.')

qc_lookup = dict(
    zip(
        notebook04_qc['metric'].astype(str),
        notebook04_qc['value'],
    )
)

if 'Complete biallelic core SNPs retained' not in qc_lookup:
    raise ValueError(
        'Notebook 04 QC does not contain the retained-SNP count.'
    )

retained_snps = int(float(qc_lookup['Complete biallelic core SNPs retained']))

if retained_snps != EXPECTED_RELATEDNESS_SNPS:
    raise ValueError(
        f'Expected K to be based on {EXPECTED_RELATEDNESS_SNPS} SNPs; '
        f'Notebook 04 QC reports {retained_snps}.'
    )

input_summary = pd.DataFrame([
    {'component': 'Pathogens with exact ceftazidime MIC', 'value': len(pathogen_index)},
    {'component': 'Candidate acquired beta-lactamase genes', 'value': len(acquired_gene_names)},
    {'component': 'K dimensions', 'value': f'{K.shape[0]} x {K.shape[1]}'},
    {'component': 'Biallelic core SNPs used for K', 'value': retained_snps},
])

display(input_summary)

print('Input alignment and QC passed.')
print(
    'Transition: Cell 01.4 will reconstruct the 176 blaTEM-1-only pathogens '
    'and the same 16 upper-MIC outliers.'
)


In [ ]:
#@title Cell 01.4 - Reconstruct the blaTEM-1-only cohort and upper-MIC group
# This cell applies the previous acquired-gene rule and the previous log2(MIC) 1.5 x IQR upper-outlier rule.

TEM_GENE = 'blaTEM-1'
EXPECTED_TEM_ONLY = 176
EXPECTED_HIGH_MIC = 16
EXPECTED_COMPARISON = 160

if TEM_GENE not in acquired_gene_names:
    raise ValueError(f'{TEM_GENE} is not present in the acquired-gene panel.')

other_acquired_genes = [
    gene
    for gene in acquired_gene_names
    if gene != TEM_GENE
]

tem_present = (
    acquired_presence[TEM_GENE].to_numpy(dtype=np.uint8) == 1
)

if other_acquired_genes:
    other_acquired_count = (
        acquired_presence[other_acquired_genes]
        .to_numpy(dtype=np.uint8)
        .sum(axis=1)
    )
else:
    other_acquired_count = np.zeros(
        len(acquired_presence),
        dtype=np.int64,
    )

tem_only_mask = tem_present & (other_acquired_count == 0)

if int(tem_only_mask.sum()) != EXPECTED_TEM_ONLY:
    raise ValueError(
        f'Expected {EXPECTED_TEM_ONLY} blaTEM-1-only pathogens; '
        f'found {int(tem_only_mask.sum())}.'
    )

tem_only = pathogen_index.loc[
    tem_only_mask,
    [
        'notebook03_row_index',
        'biosample',
        'assembly_accession',
        'log2_mic',
    ],
].copy()

tem_only['observed_mic'] = np.power(
    2.0,
    tem_only['log2_mic'].to_numpy(dtype=np.float64),
)

tem_only = (
    tem_only
    .sort_values(['log2_mic', 'biosample'], ascending=[False, True])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Reconstruct the previous upper-outlier rule on log2(MIC).
# ------------------------------------------------------------

q1 = float(tem_only['log2_mic'].quantile(0.25, interpolation='linear'))
q3 = float(tem_only['log2_mic'].quantile(0.75, interpolation='linear'))
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

tem_only['upper_mic_outlier'] = (
    tem_only['log2_mic'].to_numpy(dtype=np.float64) > upper_fence
)

high_mic_16 = tem_only.loc[
    tem_only['upper_mic_outlier']
].copy().reset_index(drop=True)

comparison_160 = tem_only.loc[
    ~tem_only['upper_mic_outlier']
].copy().reset_index(drop=True)

if len(high_mic_16) != EXPECTED_HIGH_MIC:
    raise ValueError(
        f'Expected {EXPECTED_HIGH_MIC} upper-MIC outliers; found {len(high_mic_16)}. '
        'Stop here and verify the outlier implementation against the previous analysis.'
    )

if len(comparison_160) != EXPECTED_COMPARISON:
    raise ValueError(
        f'Expected {EXPECTED_COMPARISON} remaining pathogens; found {len(comparison_160)}.'
    )

# ------------------------------------------------------------
# Save fixed empirical groups for the new project.
# ------------------------------------------------------------

HIGH_MIC_PATH = HIGH_MIC_DIRECTORY / '01_high_MIC_16_pathogens.csv'
COMPARISON_PATH = COMPARISON_DIRECTORY / '01_comparison_160_pathogens.csv'
TEM_ONLY_PATH = TABLE_DIRECTORY / '01_blaTEM-1_only_176_pathogens.csv'

high_mic_16.to_csv(HIGH_MIC_PATH, index=False)
comparison_160.to_csv(COMPARISON_PATH, index=False)
tem_only.to_csv(TEM_ONLY_PATH, index=False)

cohort_summary = pd.DataFrame([
    {'metric': 'blaTEM-1-only pathogens', 'value': len(tem_only)},
    {'metric': 'Q1 log2(MIC)', 'value': q1},
    {'metric': 'Q3 log2(MIC)', 'value': q3},
    {'metric': 'IQR log2(MIC)', 'value': iqr},
    {'metric': 'Upper outlier fence log2(MIC)', 'value': upper_fence},
    {'metric': 'Upper outlier fence MIC mg/L', 'value': float(2.0 ** upper_fence)},
    {'metric': 'Upper-MIC pathogens', 'value': len(high_mic_16)},
    {'metric': 'Remaining pathogens', 'value': len(comparison_160)},
])

display(cohort_summary)

print('\nUpper-MIC pathogens:')
display(
    high_mic_16[
        ['biosample', 'assembly_accession', 'log2_mic', 'observed_mic']
    ]
)

print(f'\nSaved: {HIGH_MIC_PATH}')
print(f'Saved: {COMPARISON_PATH}')
print(f'Saved: {TEM_ONLY_PATH}')
print(
    'Transition: Cell 01.5 will subset K and calculate pairwise relatedness '
    'within and between the empirical groups.'
)


In [ ]:
#@title Cell 01.5 - Construct K subsets and pairwise relatedness summaries
# This cell compares pairwise K values within the 16, within the 160 and across the two groups.

tem_rows = tem_only['notebook03_row_index'].to_numpy(dtype=np.int64)

K_tem = K[np.ix_(tem_rows, tem_rows)].astype(np.float64)

if K_tem.shape != (EXPECTED_TEM_ONLY, EXPECTED_TEM_ONLY):
    raise ValueError('blaTEM-1-only K subset has unexpected dimensions.')

high_local = np.flatnonzero(
    tem_only['upper_mic_outlier'].to_numpy(dtype=bool)
)
comparison_local = np.flatnonzero(
    ~tem_only['upper_mic_outlier'].to_numpy(dtype=bool)
)

if len(high_local) != EXPECTED_HIGH_MIC:
    raise ValueError('High-MIC local index count is incorrect.')
if len(comparison_local) != EXPECTED_COMPARISON:
    raise ValueError('Comparison local index count is incorrect.')

K_high = K_tem[np.ix_(high_local, high_local)]
K_comparison = K_tem[np.ix_(comparison_local, comparison_local)]


def upper_triangle_values(matrix):
    row_index, column_index = np.triu_indices(matrix.shape[0], k=1)
    return matrix[row_index, column_index]


high_pairwise = upper_triangle_values(K_high)
comparison_pairwise = upper_triangle_values(K_comparison)
all_pairwise = upper_triangle_values(K_tem)
cross_pairwise = K_tem[np.ix_(high_local, comparison_local)].reshape(-1)


def relatedness_summary(label, values):
    values = np.asarray(values, dtype=np.float64)
    return {
        'comparison': label,
        'n_pairs': int(len(values)),
        'mean_relatedness': float(np.mean(values)),
        'median_relatedness': float(np.median(values)),
        'standard_deviation': float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
        'minimum_relatedness': float(np.min(values)),
        'maximum_relatedness': float(np.max(values)),
    }


pairwise_summary = pd.DataFrame([
    relatedness_summary('Within upper-MIC 16', high_pairwise),
    relatedness_summary('Within remaining 160', comparison_pairwise),
    relatedness_summary('Upper-MIC 16 versus remaining 160', cross_pairwise),
    relatedness_summary('Within all blaTEM-1-only 176', all_pairwise),
])

PAIRWISE_SUMMARY_PATH = TABLE_DIRECTORY / '01_pairwise_relatedness_summary.csv'
HIGH_MATRIX_PATH = RELATEDNESS_DIRECTORY / '01_high_MIC_16_relatedness_matrix.csv.gz'
HIGH_PAIR_PATH = RELATEDNESS_DIRECTORY / '01_high_MIC_16_pairwise_relatedness.csv'

pairwise_summary.to_csv(PAIRWISE_SUMMARY_PATH, index=False)

high_ids = tem_only.iloc[high_local]['biosample'].astype(str).tolist()

pd.DataFrame(
    K_high,
    index=high_ids,
    columns=high_ids,
).to_csv(
    HIGH_MATRIX_PATH,
    compression='gzip',
)

pair_rows = []
for i in range(len(high_local)):
    for j in range(i + 1, len(high_local)):
        left = tem_only.iloc[high_local[i]]
        right = tem_only.iloc[high_local[j]]
        pair_rows.append({
            'biosample_1': str(left['biosample']),
            'biosample_2': str(right['biosample']),
            'log2_mic_1': float(left['log2_mic']),
            'log2_mic_2': float(right['log2_mic']),
            'relatedness_K': float(K_high[i, j]),
        })

high_pairwise_table = pd.DataFrame(pair_rows)
high_pairwise_table.to_csv(HIGH_PAIR_PATH, index=False)

display(pairwise_summary)

print(f'Saved: {PAIRWISE_SUMMARY_PATH}')
print(f'Saved: {HIGH_MATRIX_PATH}')
print(f'Saved: {HIGH_PAIR_PATH}')
print(
    'Transition: Cell 01.6 will test whether the mean pairwise relatedness '
    'among the 16 is greater than expected for random groups of 16.'
)


In [ ]:
#@title Cell 01.6 - Permutation test of upper-MIC chromosomal clustering
# This cell compares the observed mean pairwise K value with random 16-pathogen groups from the same 176-pathogen cohort.

N_PERMUTATIONS = 10000
RANDOM_SEED = 20260911

observed_mean_relatedness = float(np.mean(high_pairwise))

rng = np.random.default_rng(RANDOM_SEED)
permuted_mean_relatedness = np.empty(N_PERMUTATIONS, dtype=np.float64)

for permutation_index in range(N_PERMUTATIONS):
    sampled = rng.choice(
        EXPECTED_TEM_ONLY,
        size=EXPECTED_HIGH_MIC,
        replace=False,
    )

    sampled_matrix = K_tem[np.ix_(sampled, sampled)]
    permuted_mean_relatedness[permutation_index] = float(
        np.mean(upper_triangle_values(sampled_matrix))
    )

n_equal_or_greater = int(
    np.sum(permuted_mean_relatedness >= observed_mean_relatedness)
)

empirical_p = (
    1 + n_equal_or_greater
) / (
    1 + N_PERMUTATIONS
)

percentile = float(
    100.0 * np.mean(permuted_mean_relatedness <= observed_mean_relatedness)
)

permutation_summary = pd.DataFrame([
    {'metric': 'blaTEM-1-only pathogens', 'value': EXPECTED_TEM_ONLY},
    {'metric': 'Upper-MIC group size', 'value': EXPECTED_HIGH_MIC},
    {'metric': 'Observed mean pairwise relatedness', 'value': observed_mean_relatedness},
    {'metric': 'Permutations', 'value': N_PERMUTATIONS},
    {'metric': 'Mean random-group relatedness', 'value': float(np.mean(permuted_mean_relatedness))},
    {'metric': '95th percentile random-group relatedness', 'value': float(np.quantile(permuted_mean_relatedness, 0.95))},
    {'metric': '99th percentile random-group relatedness', 'value': float(np.quantile(permuted_mean_relatedness, 0.99))},
    {'metric': 'Observed percentile among random groups', 'value': percentile},
    {'metric': 'Random groups >= observed', 'value': n_equal_or_greater},
    {'metric': 'One-sided empirical p-value', 'value': empirical_p},
])

PERMUTATION_DISTRIBUTION_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '01_high_MIC_16_pairwise_relatedness_permutation_distribution.csv.gz'
)
PERMUTATION_SUMMARY_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '01_high_MIC_16_pairwise_relatedness_permutation_summary.csv'
)
PERMUTATION_FIGURE_PATH = (
    FIGURE_DIRECTORY
    / '01_high_MIC_16_pairwise_relatedness_permutation.png'
)

pd.DataFrame({
    'permutation': np.arange(1, N_PERMUTATIONS + 1),
    'mean_pairwise_relatedness': permuted_mean_relatedness,
}).to_csv(
    PERMUTATION_DISTRIBUTION_PATH,
    index=False,
    compression='gzip',
)

permutation_summary.to_csv(PERMUTATION_SUMMARY_PATH, index=False)

display(permutation_summary)

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(permuted_mean_relatedness, bins=40)
ax.axvline(
    observed_mean_relatedness,
    linewidth=2,
    label=f'Observed = {observed_mean_relatedness:.4f}',
)
ax.set_xlabel('Mean pairwise genome-wide relatedness')
ax.set_ylabel('Random groups of 16')
ax.set_title('Upper-MIC blaTEM-1-only pathogens: clustering permutation test')
ax.legend()
fig.tight_layout()
fig.savefig(PERMUTATION_FIGURE_PATH, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved: {PERMUTATION_DISTRIBUTION_PATH}')
print(f'Saved: {PERMUTATION_SUMMARY_PATH}')
print(f'Saved: {PERMUTATION_FIGURE_PATH}')
print(
    'Transition: Cell 01.7 will place the 16 upper-MIC pathogens within the '
    'chromosomal-relatedness structure of all 176 blaTEM-1-only pathogens.'
)


In [ ]:
#@title Cell 01.7 - Show the 16 within the relatedness structure of all 176 pathogens
# This cell creates a two-coordinate spectral representation of the blaTEM-1-only K submatrix for descriptive visualization.

eigenvalues, eigenvectors = np.linalg.eigh(K_tem)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

minimum_eigenvalue = float(eigenvalues.min())
if minimum_eigenvalue < -1e-8:
    raise ValueError(
        'The blaTEM-1-only K submatrix has a materially negative eigenvalue: '
        f'{minimum_eigenvalue}'
    )

positive_eigenvalues = np.clip(eigenvalues, 0.0, None)
positive_trace = float(np.sum(positive_eigenvalues))

if positive_trace <= 0:
    raise ValueError('The blaTEM-1-only K submatrix has no positive spectral trace.')

coordinates = (
    eigenvectors[:, :2]
    * np.sqrt(positive_eigenvalues[:2])[None, :]
)

coordinate_1_fraction = float(positive_eigenvalues[0] / positive_trace)
coordinate_2_fraction = float(positive_eigenvalues[1] / positive_trace)

coordinate_table = tem_only.copy()
coordinate_table['relatedness_coordinate_1'] = coordinates[:, 0]
coordinate_table['relatedness_coordinate_2'] = coordinates[:, 1]
coordinate_table['coordinate_1_positive_trace_fraction'] = coordinate_1_fraction
coordinate_table['coordinate_2_positive_trace_fraction'] = coordinate_2_fraction

COORDINATE_TABLE_PATH = (
    RELATEDNESS_DIRECTORY
    / '01_blaTEM-1_only_176_relatedness_coordinates.csv'
)
COORDINATE_FIGURE_PATH = (
    FIGURE_DIRECTORY
    / '01_blaTEM-1_only_176_relatedness_coordinates.png'
)

coordinate_table.to_csv(COORDINATE_TABLE_PATH, index=False)

fig, ax = plt.subplots(figsize=(8, 7))

remaining_mask = ~coordinate_table['upper_mic_outlier'].to_numpy(dtype=bool)
high_mask = coordinate_table['upper_mic_outlier'].to_numpy(dtype=bool)

ax.scatter(
    coordinates[remaining_mask, 0],
    coordinates[remaining_mask, 1],
    s=28,
    alpha=0.65,
    label='Remaining 160',
)

ax.scatter(
    coordinates[high_mask, 0],
    coordinates[high_mask, 1],
    s=65,
    marker='x',
    linewidths=1.8,
    label='Upper-MIC 16',
)

ax.set_xlabel(
    f'Genome-wide relatedness coordinate 1 '
    f'({100.0 * coordinate_1_fraction:.1f}% of positive trace)'
)
ax.set_ylabel(
    f'Genome-wide relatedness coordinate 2 '
    f'({100.0 * coordinate_2_fraction:.1f}% of positive trace)'
)
ax.set_title('blaTEM-1-only pathogens: genome-wide chromosomal relatedness')
ax.legend()
fig.tight_layout()
fig.savefig(COORDINATE_FIGURE_PATH, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved: {COORDINATE_TABLE_PATH}')
print(f'Saved: {COORDINATE_FIGURE_PATH}')
print(
    'Transition: Cell 01.8 will examine the relatedness structure within the '
    '16 upper-MIC pathogens themselves.'
)


In [ ]:
#@title Cell 01.8 - Examine relatedness structure within the 16 upper-MIC pathogens
# This cell derives distances from K and uses descriptive hierarchical clustering to show whether the 16 form one compact group or several backgrounds.

high_metadata = tem_only.iloc[high_local].copy().reset_index(drop=True)
high_ids = high_metadata['biosample'].astype(str).tolist()

# For a Gram-type relatedness matrix, squared Euclidean distance in the
# underlying feature space is K_ii + K_jj - 2 K_ij.
K_high_diagonal = np.diag(K_high)
distance_squared = (
    K_high_diagonal[:, None]
    + K_high_diagonal[None, :]
    - 2.0 * K_high
)

distance_squared = np.clip(distance_squared, 0.0, None)
distance_matrix = np.sqrt(distance_squared)
np.fill_diagonal(distance_matrix, 0.0)

if not np.allclose(distance_matrix, distance_matrix.T, rtol=1e-10, atol=1e-10):
    raise ValueError('Derived distance matrix is not symmetric.')

condensed_distance = squareform(distance_matrix, checks=False)
linkage_matrix = linkage(condensed_distance, method='average')
leaf_order = leaves_list(linkage_matrix).astype(int)

ordered_metadata = high_metadata.iloc[leaf_order].copy().reset_index(drop=True)
ordered_ids = ordered_metadata['biosample'].astype(str).tolist()
ordered_K = K_high[np.ix_(leaf_order, leaf_order)]

DISTANCE_MATRIX_PATH = (
    RELATEDNESS_DIRECTORY
    / '01_high_MIC_16_K_derived_distance_matrix.csv.gz'
)
CLUSTER_ORDER_PATH = (
    RELATEDNESS_DIRECTORY
    / '01_high_MIC_16_hierarchical_cluster_order.csv'
)
LINKAGE_PATH = (
    RELATEDNESS_DIRECTORY
    / '01_high_MIC_16_average_linkage.csv'
)
DENDROGRAM_PATH = (
    FIGURE_DIRECTORY
    / '01_high_MIC_16_relatedness_dendrogram.png'
)
HEATMAP_PATH = (
    FIGURE_DIRECTORY
    / '01_high_MIC_16_relatedness_heatmap.png'
)

pd.DataFrame(
    distance_matrix,
    index=high_ids,
    columns=high_ids,
).to_csv(
    DISTANCE_MATRIX_PATH,
    compression='gzip',
)

ordered_metadata.insert(
    0,
    'cluster_order',
    np.arange(1, len(ordered_metadata) + 1),
)
ordered_metadata.to_csv(CLUSTER_ORDER_PATH, index=False)

pd.DataFrame(
    linkage_matrix,
    columns=['cluster_1', 'cluster_2', 'distance', 'cluster_size'],
).to_csv(LINKAGE_PATH, index=False)

# Dendrogram.
fig, ax = plt.subplots(figsize=(11, 6))
dendrogram(
    linkage_matrix,
    labels=high_ids,
    leaf_rotation=90,
    ax=ax,
)
ax.set_ylabel('K-derived chromosomal distance')
ax.set_title('Upper-MIC 16: genome-wide chromosomal relatedness dendrogram')
fig.tight_layout()
fig.savefig(DENDROGRAM_PATH, dpi=300, bbox_inches='tight')
plt.show()

# Relatedness heatmap in dendrogram order.
fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(ordered_K, aspect='auto')
ax.set_xticks(np.arange(len(ordered_ids)))
ax.set_yticks(np.arange(len(ordered_ids)))
ax.set_xticklabels(ordered_ids, rotation=90)
ax.set_yticklabels(ordered_ids)
ax.set_xlabel('BioSample')
ax.set_ylabel('BioSample')
ax.set_title('Upper-MIC 16: genome-wide relatedness K')
fig.colorbar(image, ax=ax, label='K relatedness')
fig.tight_layout()
fig.savefig(HEATMAP_PATH, dpi=300, bbox_inches='tight')
plt.show()

print('\nUpper-MIC pathogens in dendrogram order:')
display(
    ordered_metadata[
        ['cluster_order', 'biosample', 'assembly_accession', 'log2_mic', 'observed_mic']
    ]
)

print(f'\nSaved: {DISTANCE_MATRIX_PATH}')
print(f'Saved: {CLUSTER_ORDER_PATH}')
print(f'Saved: {LINKAGE_PATH}')
print(f'Saved: {DENDROGRAM_PATH}')
print(f'Saved: {HEATMAP_PATH}')
print(
    'Transition: Cell 01.9 will save final QC, provenance, the analysis summary '
    'and one complete Notebook 01 output ZIP.'
)


In [ ]:
#@title Cell 01.9 - Final QC, summary, provenance and output ZIP
# This cell validates the Notebook 01 outputs, saves provenance and creates one complete output ZIP.

ANALYSIS_SUMMARY_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '01_analysis_summary.csv'
)
QC_SUMMARY_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '01_qc_summary.csv'
)
MANIFEST_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '01_manifest.json'
)
FINAL_ZIP_PATH = (
    STATISTICAL_OUTPUT_DIRECTORY
    / '01_TEM1_High_MIC_Chromosomal_Clustering_outputs.zip'
)

if empirical_p < 0.05:
    clustering_interpretation = (
        'The 16 upper-MIC pathogens have greater mean pairwise genome-wide '
        'relatedness than expected for random groups of 16 from the same '
        'blaTEM-1-only cohort.'
    )
else:
    clustering_interpretation = (
        'The 16 upper-MIC pathogens do not show significantly greater mean '
        'pairwise genome-wide relatedness than random groups of 16 from the '
        'same blaTEM-1-only cohort.'
    )

analysis_summary = pd.DataFrame([
    {'metric': 'blaTEM-1-only pathogens', 'value': EXPECTED_TEM_ONLY},
    {'metric': 'Upper-MIC pathogens', 'value': EXPECTED_HIGH_MIC},
    {'metric': 'Remaining pathogens', 'value': EXPECTED_COMPARISON},
    {'metric': 'Upper-outlier fence log2(MIC)', 'value': upper_fence},
    {'metric': 'Observed upper-MIC mean pairwise relatedness', 'value': observed_mean_relatedness},
    {'metric': 'Random-group mean pairwise relatedness', 'value': float(np.mean(permuted_mean_relatedness))},
    {'metric': 'One-sided empirical p-value', 'value': empirical_p},
    {'metric': 'Permutation count', 'value': N_PERMUTATIONS},
    {'metric': 'Interpretation', 'value': clustering_interpretation},
])

analysis_summary.to_csv(ANALYSIS_SUMMARY_PATH, index=False)

qc_summary = pd.DataFrame([
    {'metric': 'Previous-project pathogen rows', 'value': len(pathogen_index)},
    {'metric': 'Previous-project K rows', 'value': K.shape[0]},
    {'metric': 'Previous-project K columns', 'value': K.shape[1]},
    {'metric': 'SNPs used for K', 'value': retained_snps},
    {'metric': 'blaTEM-1-only count reproduced', 'value': len(tem_only)},
    {'metric': 'Upper-MIC outlier count reproduced', 'value': len(high_mic_16)},
    {'metric': 'Remaining count reproduced', 'value': len(comparison_160)},
    {'metric': 'K symmetric', 'value': bool(np.allclose(K, K.T, rtol=1e-10, atol=1e-10))},
    {'metric': 'K finite', 'value': bool(np.isfinite(K).all())},
    {'metric': 'Permutation random seed', 'value': RANDOM_SEED},
])

qc_summary.to_csv(QC_SUMMARY_PATH, index=False)

manifest = {
    'notebook': '01_TEM1_High_MIC_Chromosomal_Clustering.ipynb',
    'project': 'Ceftazidime_Chromosomal_Evolution',
    'previous_project': 'Genome_MIC_AMR_Emergence',
    'phenotype': 'observed continuous ceftazidime log2(MIC)',
    'blaTEM-1_only_definition': (
        'blaTEM-1 present and no other candidate acquired beta-lactamase '
        'in the Notebook 05 acquired-gene panel present'
    ),
    'upper_mic_rule': 'log2(MIC) > Q3 + 1.5*IQR within blaTEM-1-only pathogens',
    'expected_counts': {
        'blaTEM-1_only': EXPECTED_TEM_ONLY,
        'upper_MIC': EXPECTED_HIGH_MIC,
        'remaining': EXPECTED_COMPARISON,
    },
    'relatedness_source': {
        'matrix': str(RELATEDNESS_MATRIX_PATH),
        'SNPs': int(retained_snps),
        'definition': 'Notebook 04 genome-wide chromosomal relatedness matrix K',
    },
    'primary_test': (
        'one-sided permutation test of mean pairwise K relatedness among '
        'the 16 upper-MIC pathogens'
    ),
    'permutations': int(N_PERMUTATIONS),
    'random_seed': int(RANDOM_SEED),
    'input_sha256': {
        'acquired_presence': file_sha256(ACQUIRED_PRESENCE_PATH),
        'pathogen_index': file_sha256(PATHOGEN_INDEX_PATH),
        'relatedness_matrix': file_sha256(RELATEDNESS_MATRIX_PATH),
        'notebook04_qc': file_sha256(NOTEBOOK04_QC_PATH),
    },
}

with open(MANIFEST_PATH, 'w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2)

output_files = [
    HIGH_MIC_PATH,
    COMPARISON_PATH,
    TEM_ONLY_PATH,
    PAIRWISE_SUMMARY_PATH,
    HIGH_MATRIX_PATH,
    HIGH_PAIR_PATH,
    PERMUTATION_DISTRIBUTION_PATH,
    PERMUTATION_SUMMARY_PATH,
    PERMUTATION_FIGURE_PATH,
    COORDINATE_TABLE_PATH,
    COORDINATE_FIGURE_PATH,
    DISTANCE_MATRIX_PATH,
    CLUSTER_ORDER_PATH,
    LINKAGE_PATH,
    DENDROGRAM_PATH,
    HEATMAP_PATH,
    ANALYSIS_SUMMARY_PATH,
    QC_SUMMARY_PATH,
    MANIFEST_PATH,
]

missing_outputs = [
    str(path)
    for path in output_files
    if not path.exists()
]

if missing_outputs:
    raise FileNotFoundError(
        'Notebook 01 expected output(s) are missing:\n'
        + '\n'.join(missing_outputs)
    )

with zipfile.ZipFile(
    FINAL_ZIP_PATH,
    'w',
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in output_files:
        archive.write(
            path,
            arcname=str(path.relative_to(PROJECT_ROOT)),
        )

display(analysis_summary)

print('\nQC summary:')
display(qc_summary)

print(f'\nSaved: {ANALYSIS_SUMMARY_PATH}')
print(f'Saved: {QC_SUMMARY_PATH}')
print(f'Saved: {MANIFEST_PATH}')
print(f'Saved: {FINAL_ZIP_PATH}')
print('\nNotebook 01 complete.')
